# Ariane133 PPO Proxy-Cost Run on Kaggle

Notebook này chạy toàn bộ flow RL trên Kaggle theo objective đúng:

`proxy_cost = 1.0 * wirelength_cost + 0.5 * density_cost + 0.5 * congestion_cost`

Thứ tự chạy:

1. clone hoặc cập nhật repo
2. cài dependency Python tối thiểu
3. tạo `initial_safe.plc` nếu chưa có
4. train PPO
5. evaluate policy
6. đo lại proxy cost
7. export Tcl artifact để tải về local

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdkhoa130091/DATN.git"
REPO_DIR = Path("/kaggle/working/DATN")

if REPO_DIR.exists():
    print("Repo already exists:", REPO_DIR)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

print("Using repo:", REPO_DIR)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "absl-py",
        "gymnasium==1.0.0",
        "PyYAML",
        "pandas",
        "networkx",
    ],
    check=True,
)

import torch
import numpy
import pandas
import yaml
import gymnasium
import absl

print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
NETLIST = "MacroPlacement/Flows/NanGate45/ariane133/netlist/output_CT_Grouping/netlist.pb.txt"
INITIAL_PLC = "MacroPlacement/Flows/NanGate45/ariane133/netlist/output_CT_Grouping/initial.plc"
INITIAL_SAFE_PLC = "MacroPlacement/Flows/NanGate45/ariane133/netlist/output_CT_Grouping/initial_safe.plc"
FIX_PLC = "openroad_docker_lab/scripts/fix_plc.py"

initial_safe_path = REPO_DIR / INITIAL_SAFE_PLC
if initial_safe_path.exists():
    print("initial_safe.plc already exists")
else:
    subprocess.run(
        [
            sys.executable,
            FIX_PLC,
            "--input",
            INITIAL_PLC,
            "--output",
            INITIAL_SAFE_PLC,
            "--margin_grid_cells",
            "1",
        ],
        cwd=REPO_DIR,
        check=True,
    )

print("netlist exists:", (REPO_DIR / NETLIST).exists())
print("initial_safe exists:", (REPO_DIR / INITIAL_SAFE_PLC).exists())

In [ ]:
SEEDS = [1]
EPISODES = 20
ROLLOUT_EPISODES = 4
MAX_MACROS = 133
MAX_NODES = 1024
MAX_EDGES = 4000
MAX_GRID = 32
BATCH_SIZE = 1
WIRELENGTH_WEIGHT = 1.0
DENSITY_WEIGHT = 0.5
CONGESTION_WEIGHT = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_OUT_DIR = "experiments/kaggle_proxy"

print({
    "seeds": SEEDS,
    "episodes": EPISODES,
    "rollout_episodes": ROLLOUT_EPISODES,
    "max_macros": MAX_MACROS,
    "max_nodes": MAX_NODES,
    "max_edges": MAX_EDGES,
    "max_grid": MAX_GRID,
    "batch_size": BATCH_SIZE,
    "device": DEVICE,
    "wirelength_weight": WIRELENGTH_WEIGHT,
    "density_weight": DENSITY_WEIGHT,
    "congestion_weight": CONGESTION_WEIGHT,
})

In [ ]:
def run_cmd(cmd, cwd=REPO_DIR):
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

for seed in SEEDS:
    out_dir = f"{BASE_OUT_DIR}/seed_{seed}"
    run_cmd([
        sys.executable,
        "rl_macroplacement_agent/scripts/train_ppo.py",
        "--netlist", NETLIST,
        "--init_plc", INITIAL_SAFE_PLC,
        "--out_dir", out_dir,
        "--episodes", str(EPISODES),
        "--rollout_episodes", str(ROLLOUT_EPISODES),
        "--max_macros", str(MAX_MACROS),
        "--max_nodes", str(MAX_NODES),
        "--max_edges", str(MAX_EDGES),
        "--max_grid", str(MAX_GRID),
        "--wirelength_weight", str(WIRELENGTH_WEIGHT),
        "--density_weight", str(DENSITY_WEIGHT),
        "--congestion_weight", str(CONGESTION_WEIGHT),
        "--batch_size", str(BATCH_SIZE),
        "--seed", str(seed),
        "--device", DEVICE,
    ])

In [ ]:
import pandas as pd

rows = []
for seed in SEEDS:
    summary_path = REPO_DIR / BASE_OUT_DIR / f"seed_{seed}" / "alphachip_like_train_summary.json"
    data = json.loads(summary_path.read_text())
    last = data["last_episode"]
    rows.append({
        "seed": seed,
        "best_cost": data["best_cost"],
        "final_cost": last["final_cost"],
        "initial_cost": last["initial_cost"],
        "wirelength_cost": last.get("wirelength_cost"),
        "density_cost": last.get("density_cost"),
        "congestion_cost": last.get("congestion_cost"),
        "runtime_sec": data["train_runtime_sec"],
    })

train_df = pd.DataFrame(rows)
train_df

In [ ]:
EVAL_SEED = SEEDS[0]
MODEL = f"{BASE_OUT_DIR}/seed_{EVAL_SEED}/alphachip_like_actor_critic.pt"
EVAL_OUT_DIR = f"{BASE_OUT_DIR}/seed_{EVAL_SEED}_eval"

run_cmd([
    sys.executable,
    "rl_macroplacement_agent/scripts/eval_policy.py",
    "--model", MODEL,
    "--netlist", NETLIST,
    "--init_plc", INITIAL_SAFE_PLC,
    "--out_dir", EVAL_OUT_DIR,
    "--max_macros", str(MAX_MACROS),
    "--max_nodes", str(MAX_NODES),
    "--max_edges", str(MAX_EDGES),
    "--max_grid", str(MAX_GRID),
    "--wirelength_weight", str(WIRELENGTH_WEIGHT),
    "--density_weight", str(DENSITY_WEIGHT),
    "--congestion_weight", str(CONGESTION_WEIGHT),
    "--device", DEVICE,
    "--deterministic",
])

In [ ]:
FINAL_PLC = f"{EVAL_OUT_DIR}/alphachip_like_final.plc"
PROXY_JSON = f"{EVAL_OUT_DIR}/proxy_eval.json"

run_cmd([
    sys.executable,
    "rl_macroplacement_agent/scripts/eval_proxy.py",
    "--netlist", NETLIST,
    "--plc", FINAL_PLC,
    "--out", PROXY_JSON,
    "--wirelength_weight", str(WIRELENGTH_WEIGHT),
    "--density_weight", str(DENSITY_WEIGHT),
    "--congestion_weight", str(CONGESTION_WEIGHT),
])

proxy_data = json.loads((REPO_DIR / PROXY_JSON).read_text())
proxy_data

In [ ]:
train_summary = json.loads((REPO_DIR / BASE_OUT_DIR / f"seed_{EVAL_SEED}" / "alphachip_like_train_summary.json").read_text())
eval_summary = json.loads((REPO_DIR / EVAL_OUT_DIR / "alphachip_like_eval_summary.json").read_text())

comparison = pd.DataFrame([
    {
        "name": "train_summary_best",
        "proxy_cost": train_summary["best_cost"],
    },
    {
        "name": "eval_final",
        "proxy_cost": eval_summary["cost"],
    },
    {
        "name": "eval_proxy_recomputed",
        "proxy_cost": proxy_data["proxy_cost"],
    },
])
comparison

In [ ]:
EXPORT_DIR = REPO_DIR / EVAL_OUT_DIR / "openroad_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RAW_TCL = str(Path(EVAL_OUT_DIR) / "openroad_export" / "place_instance_raw.tcl")
CLAMPED_TCL = str(Path(EVAL_OUT_DIR) / "openroad_export" / "place_instance_clamped.tcl")
OPENROAD_TCL = str(Path(EVAL_OUT_DIR) / "openroad_export" / "openroad_macro.tcl")
DEF_FILE = "MacroPlacement/Flows/NanGate45/ariane133/def/ariane133_fp.def"

run_cmd([
    sys.executable,
    "MacroPlacement/Flows/util/plc_pb_to_placement_tcl.py",
    FINAL_PLC,
    NETLIST,
    RAW_TCL,
])

run_cmd([
    sys.executable,
    "rl_macroplacement_agent/scripts/clamp_place_tcl_to_def_core.py",
    "--in_tcl", RAW_TCL,
    "--pb", NETLIST,
    "--def_file", DEF_FILE,
    "--out_tcl", CLAMPED_TCL,
])

run_cmd([
    sys.executable,
    "rl_macroplacement_agent/scripts/plc_to_openroad_tcl.py",
    "--in_tcl", CLAMPED_TCL,
    "--out_tcl", OPENROAD_TCL,
    "--mode", "place_macro",
    "--escape_brackets",
])

print("Exported files:")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path)

In [ ]:
bundle = REPO_DIR / EVAL_OUT_DIR / "kaggle_artifacts_seed_bundle.zip"
subprocess.run([
    "zip",
    "-r",
    str(bundle),
    str(REPO_DIR / BASE_OUT_DIR / f"seed_{EVAL_SEED}"),
    str(REPO_DIR / EVAL_OUT_DIR),
], check=True)
print("Bundle:", bundle)